# 🎯 Day 02a: Path & Query Parameters — Handling Input Like a Pro

---

## 🎯 What You'll Master Today
- Path parameters with type validation
- Query parameters with defaults and constraints
- Request body with Pydantic models
- Form data and file uploads
- Combining all input types in one endpoint

---

## 📦 Three Ways Data Comes In

```
╔═══════════════════════════════════════════════════════════════════╗
║  HOW DATA ENTERS YOUR API                                        ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. PATH PARAMETERS — In the URL path                           ║
║     GET /users/42         → user_id = 42                        ║
║     GET /posts/hello-world → slug = "hello-world"               ║
║                                                                   ║
║  2. QUERY PARAMETERS — After the ? in URL                       ║
║     GET /users?page=2&limit=20 → page=2, limit=20              ║
║     GET /search?q=python       → q="python"                     ║
║                                                                   ║
║  3. REQUEST BODY — JSON in POST/PUT/PATCH                       ║
║     POST /users                                                   ║
║     {"name": "Sarthak", "email": "s@mail.com"}                 ║
║                                                                   ║
║  RULE: Path = identity, Query = filtering, Body = data          ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# PATH PARAMETERS — Identity of a resource
# ============================================================
from fastapi import FastAPI, Path, Query, HTTPException
from fastapi.testclient import TestClient
from enum import Enum

app = FastAPI()

# Basic path parameter — type validated automatically
@app.get("/users/{user_id}")
async def get_user(user_id: int):  # FastAPI converts string to int
    return {"user_id": user_id}

# With constraints using Path()
@app.get("/items/{item_id}")
async def get_item(
    item_id: int = Path(
        ...,  # ... means required
        title="Item ID",
        description="The ID of the item to get",
        ge=1,       # greater than or equal to 1
        le=10000    # less than or equal to 10000
    )
):
    return {"item_id": item_id}

# Enum path parameter — restricts to specific values
class ModelName(str, Enum):
    gpt4 = "gpt-4"
    claude = "claude"
    gemini = "gemini"

@app.get("/models/{model_name}")
async def get_model(model_name: ModelName):
    return {"model": model_name.value}

# Test
client = TestClient(app)
print(client.get("/users/42").json())        # {"user_id": 42}
print(client.get("/users/abc").status_code)   # 422 — not an int!
print(client.get("/models/gpt-4").json())     # {"model": "gpt-4"}
print(client.get("/models/llama").status_code) # 422 — not in enum

In [ ]:
# ============================================================
# QUERY PARAMETERS — Filtering, sorting, pagination
# ============================================================
from typing import Optional

@app.get("/products")
async def list_products(
    # All these come from ?key=value in the URL
    page: int = Query(default=1, ge=1, description="Page number"),
    limit: int = Query(default=20, ge=1, le=100, description="Items per page"),
    sort: str = Query(default="created_at", description="Sort field"),
    order: str = Query(default="desc", regex="^(asc|desc)$"),
    search: Optional[str] = Query(default=None, min_length=2, max_length=50),
    category: Optional[str] = None,  # simple optional, no Query() needed
):
    return {
        "page": page,
        "limit": limit,
        "sort": f"{sort} {order}",
        "search": search,
        "category": category,
    }

# Test various combinations
print("Default params:")
print(client.get("/products").json())

print("\nWith filters:")
print(client.get("/products?page=2&limit=10&search=python&category=books").json())

print("\nInvalid limit (>100):")
r = client.get("/products?limit=200")
print(f"  Status: {r.status_code}")  # 422

In [ ]:
# ============================================================
# REQUEST BODY — Pydantic models for structured data
# ============================================================
from pydantic import BaseModel, Field, field_validator
from typing import Optional
from datetime import datetime

class PostCreate(BaseModel):
    title: str = Field(..., min_length=3, max_length=100)
    content: str = Field(..., min_length=10)
    tags: list[str] = Field(default=[], max_length=5)  # max 5 tags
    is_published: bool = False

class PostUpdate(BaseModel):
    """All fields optional for PATCH."""
    title: Optional[str] = Field(None, min_length=3, max_length=100)
    content: Optional[str] = Field(None, min_length=10)
    tags: Optional[list[str]] = None
    is_published: Optional[bool] = None

class PostResponse(BaseModel):
    id: int
    title: str
    content: str
    tags: list[str]
    is_published: bool
    created_at: datetime

posts_db = {}
post_counter = 0

@app.post("/posts", response_model=PostResponse, status_code=201)
async def create_post(post: PostCreate):
    global post_counter
    post_counter += 1
    new_post = {
        "id": post_counter,
        **post.model_dump(),
        "created_at": datetime.utcnow()
    }
    posts_db[post_counter] = new_post
    return new_post

@app.patch("/posts/{post_id}", response_model=PostResponse)
async def update_post(post_id: int, post: PostUpdate):
    if post_id not in posts_db:
        raise HTTPException(404, "Post not found")
    # Only update fields that were actually sent
    update_data = post.model_dump(exclude_unset=True)
    posts_db[post_id].update(update_data)
    return posts_db[post_id]

# Test
r = client.post("/posts", json={
    "title": "FastAPI Guide",
    "content": "This is my awesome post about FastAPI!",
    "tags": ["python", "fastapi"]
})
print(f"Created: {r.json()}")

# Partial update — only title
r = client.patch("/posts/1", json={"title": "Updated Title"})
print(f"Updated: {r.json()['title']}")  # content unchanged!

In [ ]:
# ============================================================
# COMBINING ALL THREE — Path + Query + Body
# ============================================================
from fastapi import Body

@app.put("/users/{user_id}/posts/{post_id}")
async def update_user_post(
    user_id: int = Path(..., ge=1),           # Path parameter
    post_id: int = Path(..., ge=1),            # Path parameter
    notify: bool = Query(default=False),        # Query parameter
    post: PostUpdate = Body(...),               # Request body
):
    return {
        "user_id": user_id,
        "post_id": post_id,
        "notify": notify,
        "update": post.model_dump(exclude_unset=True)
    }

# Test: PUT /users/5/posts/10?notify=true  +  JSON body
r = client.put(
    "/users/5/posts/10?notify=true",
    json={"title": "New Title"}
)
print(r.json())

In [ ]:
# ============================================================
# FILE UPLOAD — Form data
# ============================================================
from fastapi import File, UploadFile, Form

@app.post("/upload")
async def upload_file(
    file: UploadFile = File(...),
    description: str = Form(default="No description"),
):
    contents = await file.read()
    return {
        "filename": file.filename,
        "content_type": file.content_type,
        "size": len(contents),
        "description": description,
    }

# Multiple files
@app.post("/upload-multiple")
async def upload_multiple(files: list[UploadFile] = File(...)):
    return [
        {"filename": f.filename, "size": len(await f.read())}
        for f in files
    ]

print("📁 UploadFile gives you:")
print("  .filename, .content_type, .read(), .write(), .seek()")
print("  For large files, use .read(chunk_size) in a loop")

---

## 🗺️ Parameters Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  FASTAPI PARAMETERS QUICK REFERENCE                              ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  PATH:    /users/{user_id}  →  user_id: int                     ║
║           Constraints: Path(ge=1, le=100)                        ║
║                                                                   ║
║  QUERY:   ?page=1&limit=20  →  page: int = 1                    ║
║           Constraints: Query(ge=1, le=100, regex=...)            ║
║                                                                   ║
║  BODY:    JSON payload  →  user: UserCreate                      ║
║           Uses Pydantic BaseModel with Field()                   ║
║                                                                   ║
║  FILE:    multipart  →  file: UploadFile = File(...)             ║
║  FORM:    form data  →  name: str = Form(...)                    ║
║                                                                   ║
║  PARTIAL UPDATE: model_dump(exclude_unset=True)                  ║
║  ENUM:   class Status(str, Enum) for fixed choices              ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Path vs query parameter? | Path = resource identity (/users/1). Query = filtering (?page=2&sort=name) |
| 2 | How does FastAPI validate input? | Type hints for path/query, Pydantic models for body. Auto 422 on failure |
| 3 | How to handle partial updates? | PATCH with Optional fields + model_dump(exclude_unset=True) |
| 4 | File upload in FastAPI? | UploadFile = File(...), reads async, supports streaming |
| 5 | What is Field()? | Like Query/Path but for Pydantic model fields. Adds constraints + docs |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Path = identity, Query = filters, Body = data        │
## │  • Path/Query/Field for constraints (ge, le, regex)     │
## │  • Pydantic BaseModel validates request body             │
## │  • exclude_unset=True for PATCH partial updates          │
## │  • Enum restricts path params to fixed values            │
## │  • UploadFile for async file handling                    │
## │  • FastAPI auto-returns 422 for validation failures      │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Pydantic Deep Dive** — Advanced validation, nested models, and custom validators